# 03 — Training runner, pre-configured: `c1_ce_s6out` (E2′ living-out)

A copy of the `03_train` template with `RUN` pre-set — open, run all,
no edits needed.

**Why this run exists (E2′, v5.2 §10.3):** the primary rotation tests on
S7 (laboratory — environment, day AND person unseen). This rotation
tests C1 on **S6 (living room, person P1 seen in train)** with S7 moved
into train: a second, different cross-domain regime for the C1 baseline,
and the input for replicating the §9 domain diagnostics on a train set
whose non-bedroom environment is the laboratory instead of the living
room. Config `c1_ce_s6out.yaml` is byte-identical to `c1_ce.yaml` except
`name`/`protocol`/`split_file` — same seed 42, same budget (~2.3 h).

**Prerequisite:** `splits/p2_living.json` frozen on Git (session
`01_s6out_split.ipynb`, committed and pushed) — asserted below.

Archive the executed copy under `notebooks/runs/` as
`YYYY-MM-DD_c1_ce_s6out.ipynb`, as for every run (cell at the bottom).


## Environment setup

This notebook is not one run per file. Instead:

- one notebook = one runner
- one config = one experiment
- different people can use the same notebook with different config values

In Colab, the notebook runs on a remote VM. If the repo is not already
available there, this notebook will clone it from GitHub. It also stages
data from the shared Drive archive into `/content/data` using the
paths defined in `configs/paths.yaml`.


In [1]:
from pathlib import Path
import subprocess
import sys
import torch
import zipfile

REPO_URL = "https://github.com/FilippoIsoni/sharp-har.git"

# Locate or clone the repository root containing the sharp_har package.
# In Colab, the notebook may run from a temporary working directory.
cwd = Path.cwd().resolve()
if (cwd / "sharp_har").exists():
    REPO_DIR = cwd
elif (cwd.parent / "sharp_har").exists():
    REPO_DIR = cwd.parent
elif (cwd.parent.parent / "sharp_har").exists():
    REPO_DIR = cwd.parent.parent
else:
    REPO_DIR = Path("/content/sharp-har")
    if not REPO_DIR.exists():
        REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

# After the clone, so the file actually exists on a fresh runtime.
!pip install -q -r {REPO_DIR}/requirements.txt

sys.path.insert(0, str(REPO_DIR))

from google.colab import drive
from sharp_har.train import train_run
from sharp_har.utils import read_yaml

paths_cfg = read_yaml(REPO_DIR / "configs" / "paths.yaml")
drive_root = Path(paths_cfg["drive_root"])
stage_dir = Path(paths_cfg["stage_dir"])
CKPT_ROOT = Path(paths_cfg["ckpt_root"])

# Mount Drive unconditionally (idempotent): checkpoints are written to
# CKPT_ROOT on Drive even when the data is already staged on the VM.
drive.mount("/content/drive")

# Stage the zip archives if needed (same convention as 00_setup_smoke).
if not (stage_dir.exists() and any(stage_dir.rglob("*.txt"))):
    stage_dir.mkdir(parents=True, exist_ok=True)
    for zip_name in paths_cfg["zips"]:
        src = drive_root / zip_name
        dst = Path("/content") / zip_name
        print(f"copying {src} -> {dst}")
        subprocess.run(["cp", str(src), str(dst)], check=True)
        with zipfile.ZipFile(dst) as zf:
            zf.extractall(stage_dir)

print("Repo dir:", REPO_DIR)
print("sharp_har exists:", (REPO_DIR / "sharp_har").exists())
print("GPU available:", torch.cuda.is_available())
print("Stage dir:", stage_dir)
print("Checkpoint root:", CKPT_ROOT)


Mounted at /content/drive
copying /content/drive/MyDrive/DATASET_SHARP/doppler_traces.zip -> /content/doppler_traces.zip
copying /content/drive/MyDrive/DATASET_SHARP/doppler_traces_S4_S5.zip -> /content/doppler_traces_S4_S5.zip
Repo dir: /content/sharp-har
sharp_har exists: True
GPU available: True
Stage dir: /content/data
Checkpoint root: /content/drive/MyDrive/sharp_har_runs


In [2]:
# Readiness assert: the E2' rotation runs ONLY on the frozen split
# (§0.1). If this fails, the split session has not been run/pushed yet,
# or the clone predates it - do NOT generate the split from here.
from sharp_har.utils import read_json

split_path = REPO_DIR / "splits" / "p2_living.json"
assert split_path.exists(), (
    "splits/p2_living.json not found in the clone: freeze it first with "
    "e2_living_out/01_s6out_split.ipynb, commit and push, then restart "
    "from a fresh clone."
)
split = read_json(split_path)
assert split["protocol"] == "P2-living", split["protocol"]
print("protocol:", split["protocol"], "| split_seed:", split["split_seed"])
print("train:", len(split["train"]), "val:", len(split["val"]),
      "test:", len(split["test"]))
print("norm:", split["norm"])


protocol: P2-living | split_seed: 42
train: 80 val: 6 test: 15
norm: {'mu': 0.11912171315116504, 'sigma': 0.18088620700098512}


In [3]:
# GPU sanity check - run BEFORE launching training (see the C3 OOM incident:
# a stale process on the VM held 11.5 GiB). Expected on a clean runtime:
# no processes listed, ~0 MiB used. If not: Runtime -> Disconnect and
# delete runtime (a plain restart is NOT enough), then rerun from the top.
!nvidia-smi

import torch
free, total = torch.cuda.mem_get_info()
print(f"free: {free/2**30:.2f} GiB / total: {total/2**30:.2f} GiB")
assert free / 2**30 > 10, (
    "GPU is not clean: less than 10 GiB free before training. "
    "Something else is holding memory - delete the runtime and start over."
)
print("GPU clean - OK to launch.")


Sat Jul 18 09:58:18 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Load config and launch the run

Choose one config stem below and run the cell. The config name
determines which experiment is executed.


In [4]:
RUN = "c1_ce_s6out"  # E2' living-out rotation - do not change in-session
cfg = read_yaml(REPO_DIR / "configs" / f"{RUN}.yaml")

print("Selected run:", RUN)
print("Config summary:")
print(cfg)

out = train_run(cfg, stage_dir=stage_dir, ckpt_dir=CKPT_ROOT, repo_dir=REPO_DIR)
print("Train run finished:", out)


2026-07-18 09:58:24,627 [INFO] sharp_har.data: train: 80 traces, 50948 (window, antenna) samples (win=340, stride=100)
2026-07-18 09:58:24,640 [INFO] sharp_har.data: val: 6 traces, 956 (window, antenna) samples (win=340, stride=340)


Selected run: c1_ce_s6out
Config summary:
{'name': 'C1_s6out', 'protocol': 'P2-living', 'split_file': 'splits/p2_living.json', 'backbone': 'resnet_vb', 'd_enc': 256, 'n_att': 8, 'loss': {'type': 'ce', 'label_smoothing': 0.1}, 'adversary': {'type': None, 'target': 'ar_set', 'lambda_max': None, 'beta': None}, 'optim': {'name': 'adamw', 'lr': 0.001, 'wd': 0.0001, 'warmup_epochs': 5, 'scheduler': 'cosine'}, 'train': {'epoch_steps': 400, 'max_epochs': 40, 'batch_size': 256, 'sampler': 'uniform', 'amp': True, 'grad_clip': 1.0}, 'eval': {'select_metric': 'val_macro_f1', 'patience': 10}}


2026-07-18 10:02:21,726 [INFO] sharp_har.train: C1_s6out epoch 1/40: loss 1.5952, val macro-F1 0.5197, 0.572 s/step
2026-07-18 10:05:51,794 [INFO] sharp_har.train: C1_s6out epoch 2/40: loss 1.3643, val macro-F1 0.3135, 0.519 s/step
2026-07-18 10:09:21,484 [INFO] sharp_har.train: C1_s6out epoch 3/40: loss 1.2666, val macro-F1 0.4459, 0.519 s/step
2026-07-18 10:12:51,422 [INFO] sharp_har.train: C1_s6out epoch 4/40: loss 1.2173, val macro-F1 0.2636, 0.520 s/step
2026-07-18 10:16:21,371 [INFO] sharp_har.train: C1_s6out epoch 5/40: loss 1.2018, val macro-F1 0.1377, 0.520 s/step
2026-07-18 10:19:49,079 [INFO] sharp_har.train: C1_s6out epoch 6/40: loss 1.1390, val macro-F1 0.4026, 0.515 s/step
2026-07-18 10:23:19,177 [INFO] sharp_har.train: C1_s6out epoch 7/40: loss 1.0888, val macro-F1 0.3721, 0.520 s/step
2026-07-18 10:26:48,221 [INFO] sharp_har.train: C1_s6out epoch 8/40: loss 1.0547, val macro-F1 0.3788, 0.517 s/step
2026-07-18 10:30:16,408 [INFO] sharp_har.train: C1_s6out epoch 9/40: los

Train run finished: {'run_dir': PosixPath('/content/drive/MyDrive/sharp_har_runs/C1_s6out'), 'best_val_macro_f1': 0.7761180124223602, 'history': [{'epoch': 1, 'train_loss': 1.5951578563451767, 'val_macro_f1': 0.5197168797246527, 'lr': 0.00020050000000000002, 's_per_step': 0.5720971715450287, 'epoch_seconds': 228.83886861801147}, {'epoch': 2, 'train_loss': 1.3643474099040032, 'val_macro_f1': 0.31348817576056043, 'lr': 0.00040050000000000003, 's_per_step': 0.5186882197856904, 'epoch_seconds': 207.47528791427612}, {'epoch': 3, 'train_loss': 1.2665644708275794, 'val_macro_f1': 0.4458862876254181, 'lr': 0.0006005, 's_per_step': 0.5192463743686676, 'epoch_seconds': 207.69854974746704}, {'epoch': 4, 'train_loss': 1.2172780638933183, 'val_macro_f1': 0.26358903848454796, 'lr': 0.0008005, 's_per_step': 0.5204841357469558, 'epoch_seconds': 208.19365429878235}, {'epoch': 5, 'train_loss': 1.2017864891886711, 'val_macro_f1': 0.13766233766233765, 'lr': 0.001, 's_per_step': 0.5201864069700242, 'epoch_

## Training curves

Per-epoch diagnostics read from the run's `history.csv` (§0.4). The
panels adapt to the run: train loss and throughput always; fused val
macro-F1 with the val-selected best epoch on CE runs; learning rate.
Re-run this cell any time — it re-reads the CSV, so it also works on a
resumed or still-running run.

Reminder for this rotation: the val macro-F1 here is a **5-class**
number (H, L and W absent from val, see the split session) and val is 6
traces — selection is noisy and NOT scale-comparable to p2_lab val
numbers, nor to the 8-class test macro-F1.


In [ ]:
# Thin runner: all plotting logic lives in sharp_har.viz.
from sharp_har.viz import plot_history

run_dir = CKPT_ROOT / cfg["name"]
plot_history(run_dir);


## Archiving the definitive run notebook

This notebook stays a **clean template** (outputs cleared on Git). The
executed copy of a real run is a measured artifact and is committed
verbatim, like the gate reports:

1. When the run (or a resumed segment) ends, download the executed
   notebook (`File → Download → .ipynb`) **with its outputs**.
2. Commit it under `notebooks/runs/` as `YYYY-MM-DD_<config>.ipynb`
   (e.g. `2026-07-18_c1_ce_s6out.ipynb`); add a `_partN` suffix if one
   run spans several resumed sessions. Never edit archived outputs.
3. The run's numeric artifacts still live in `ckpt_root/<name>/`
   (`history.csv`, `run_meta.json`, checkpoints) — the archived
   notebook is the human-readable record, not a data source.

See `notebooks/runs/README.md` for the full convention.
